# SQL — Question Patterns Reference

**Companion to:** `reference_sql_syntax.ipynb` — function names, argument order, dialect differences

**Purpose:** recognize problem types and reach for the right template. Open this when you're mid-problem and need to identify the pattern.

## Quick Index

| Section | Pattern |
| :--- | :--- |
| §1 | Deduplication patterns |
| §2 | Ranking & Top N per group |
| §3 | Pivot & transpose |
| §4 | Gaps & Islands — streak detection |
| §5 | Session grouping |
| §6 | Product metrics: DAU/WAU/MAU, cohort, funnel |
| §7 | Growth metrics: WoW, MoM, rolling active users |
| §8 | Recursive patterns |
| — | Decision Guide |

---
## When to Use

| Signal words | Pattern |
| :--- | :--- |
| "remove duplicates", "keep only one" | §1 — Deduplication |
| "top N per group", "highest salary per dept" | §2 — Ranking |
| "pivot", "rows to columns", "each X as a column" | §3 — Pivot |
| "consecutive days", "streak", "unbroken sequence" | §4 — Gaps & Islands |
| "session", "group events within N minutes" | §5 — Session grouping |
| "DAU", "retention", "cohort", "funnel" | §6 — Product metrics |
| "week over week", "growth rate", "rolling users" | §7 — Growth metrics |
| "hierarchy", "org chart", "all ancestors" | §8 — Recursive CTE |

---
## §1 — Deduplication Patterns

**Signal:** "remove duplicates", "keep the earliest/latest record", "unique users only"

### §1a — DISTINCT (fully identical rows)

```sql
-- Remove fully duplicate rows across all selected columns
SELECT DISTINCT user_id, event_date FROM events;
```

### §1b — GROUP BY as deduplication

```sql
-- Deduplicate and aggregate simultaneously
SELECT user_id, COUNT(*) AS login_count, MAX(login_date) AS last_login
FROM logins
GROUP BY user_id;
```

### §1c — ROW_NUMBER() — keep specific occurrence

```sql
-- Keep the FIRST record per user (earliest date)
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at ASC) AS rn
  FROM orders
)
SELECT * FROM ranked WHERE rn = 1;

-- Keep the LAST record per user (latest date)
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) AS rn
  FROM orders
)
SELECT * FROM ranked WHERE rn = 1;
```

### §1d — NOT EXISTS / NOT IN — remove rows with a match elsewhere

```sql
-- Keep users who have never made a purchase
SELECT * FROM users u
WHERE NOT EXISTS (
  SELECT 1 FROM orders o WHERE o.user_id = u.id
);
```

### §1e — DELETE duplicates, keep one

```sql
-- Keep row with smallest id, delete the rest
DELETE p1 FROM Person p1
JOIN Person p2
  ON p1.email = p2.email
 AND p1.id > p2.id;
```

**Decision — which deduplication method:**

| Method | Use when |
| :--- | :--- |
| `DISTINCT` | All selected columns define uniqueness |
| `GROUP BY` | Need to deduplicate AND aggregate simultaneously |
| `ROW_NUMBER()` | Need to keep a specific occurrence (first, last, nth) |
| `NOT EXISTS` | Remove rows that have any match in another table |
| `DELETE + JOIN` | Physically remove duplicates from the table |

**Common mistakes:**
- Using `DISTINCT` when you need the first or last — DISTINCT doesn't control which row is kept
- Using `NOT IN` when the subquery can return NULLs — always prefer `NOT EXISTS` for safety
- Forgetting `PARTITION BY` in `ROW_NUMBER()` — without it, numbering is global not per-group

---
## §2 — Ranking & Top N per Group

**Signal:** "highest salary per department", "top 3 products per category", "second highest"

### §2a — Top N per group (ROW_NUMBER)

```sql
-- Top 3 salaries per department — no ties
WITH ranked AS (
  SELECT dept, emp_id, salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
  FROM Employee
)
SELECT dept, emp_id, salary FROM ranked WHERE rn <= 3;
```

### §2b — Top N per group with ties (DENSE_RANK)

```sql
-- Top 3 salary levels per department — ties included
WITH ranked AS (
  SELECT dept, emp_id, salary,
    DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
  FROM Employee
)
SELECT dept, emp_id, salary FROM ranked WHERE dr <= 3;
```

### §2c — Nth highest (single value)

```sql
-- Nth highest salary — handles case where fewer than N salaries exist
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  RETURN (
    SELECT DISTINCT salary FROM Employee
    ORDER BY salary DESC
    LIMIT 1 OFFSET N-1
  );
END;

-- Or with DENSE_RANK
WITH ranked AS (
  SELECT salary,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
  FROM Employee
)
SELECT MAX(salary) FROM ranked WHERE dr = N;
-- MAX handles the case where N rank doesn't exist — returns NULL
```

### §2d — Median

```sql
-- Median using ROW_NUMBER (works in MySQL)
WITH counted AS (
  SELECT id, salary, COUNT(*) OVER () AS total,
    ROW_NUMBER() OVER (ORDER BY salary) AS rn
  FROM Employee
)
SELECT AVG(salary) AS median
FROM counted
WHERE rn IN (FLOOR((total+1)/2), CEIL((total+1)/2));
```

**Decision — ROW_NUMBER vs DENSE_RANK for Top N:**
| Function | Use when |
| :--- | :--- |
| `ROW_NUMBER()` | Want exactly N rows, ties broken arbitrarily |
| `DENSE_RANK()` | Want top N distinct values, ties all included |
| `LIMIT + OFFSET` | Single value without window function support |

**Common mistakes:**
- Using `RANK()` for top N — gaps after ties mean rank 2 may not exist
- Forgetting `PARTITION BY` — without it, ranking is global not per-group
- Using `LIMIT N` directly without `DISTINCT` — may return duplicate salary values

---
## §3 — Pivot & Transpose

**Signal:** "each month as a column", "show X and Y side by side", "convert rows to columns"

### §3a — Pivot with CASE WHEN (most common)

```sql
-- Pivot: one row per product, one column per month
SELECT
  product_id,
  SUM(CASE WHEN MONTH(sale_date) = 1 THEN amount ELSE 0 END) AS Jan,
  SUM(CASE WHEN MONTH(sale_date) = 2 THEN amount ELSE 0 END) AS Feb,
  SUM(CASE WHEN MONTH(sale_date) = 3 THEN amount ELSE 0 END) AS Mar
FROM sales
GROUP BY product_id;
```

### §3b — Transpose (rows to columns, fixed columns)

```sql
-- LC 1179: Department table with Department and Revenue per quarter
SELECT
  SUM(CASE WHEN quarter = 'Q1' THEN revenue END) AS Q1_Revenue,
  SUM(CASE WHEN quarter = 'Q2' THEN revenue END) AS Q2_Revenue,
  SUM(CASE WHEN quarter = 'Q3' THEN revenue END) AS Q3_Revenue,
  SUM(CASE WHEN quarter = 'Q4' THEN revenue END) AS Q4_Revenue
FROM Department;
```

### §3c — Dynamic pivot (variable number of columns)

```sql
-- Not directly supported in MySQL — requires dynamic SQL
-- Use GROUP_CONCAT to build the CASE WHEN list dynamically
SET @sql = NULL;
SELECT GROUP_CONCAT(
  DISTINCT CONCAT(
    'SUM(CASE WHEN month = ', month, ' THEN amount ELSE 0 END) AS m', month
  )
) INTO @sql FROM sales;

SET @sql = CONCAT('SELECT product_id, ', @sql, ' FROM sales GROUP BY product_id');
PREPARE stmt FROM @sql;
EXECUTE stmt;
```

**Common mistakes:**
- Forgetting `ELSE 0` in `SUM(CASE WHEN ...)` — non-matching rows contribute NULL, not 0
- Using `MAX` vs `SUM` in pivot — use `MAX` when each cell has only one value; `SUM` when aggregating multiple rows
- Not grouping by the row identifier — always `GROUP BY` the column that defines each output row

---
## §4 — Gaps & Islands — Streak Detection

**Signal:** "consecutive days", "longest streak", "unbroken sequence", "find gaps in dates"

### §4a — Date streak detection (row number subtraction trick)

```sql
-- Trick: date - ROW_NUMBER() = constant within a consecutive streak
-- If dates are consecutive daily, the difference stays the same
WITH numbered AS (
  SELECT user_id, activity_date,
    activity_date - ROW_NUMBER() OVER (
      PARTITION BY user_id ORDER BY activity_date
    ) AS grp                           -- same value = same streak
  FROM activity
)
SELECT
  user_id,
  MIN(activity_date) AS streak_start,
  MAX(activity_date) AS streak_end,
  COUNT(*)           AS streak_length
FROM numbered
GROUP BY user_id, grp
ORDER BY user_id, streak_start;
```

### §4b — Longest streak per user

```sql
WITH numbered AS (
  SELECT user_id, activity_date,
    activity_date - ROW_NUMBER() OVER (
      PARTITION BY user_id ORDER BY activity_date
    ) AS grp
  FROM activity
),
streak_lengths AS (
  SELECT user_id, grp, COUNT(*) AS streak_len
  FROM numbered
  GROUP BY user_id, grp
)
SELECT user_id, MAX(streak_len) AS longest_streak
FROM streak_lengths
GROUP BY user_id;
```

### §4c — Integer / seat number gaps (same trick)

```sql
-- Find consecutive seat ranges
WITH numbered AS (
  SELECT seat_id,
    seat_id - ROW_NUMBER() OVER (ORDER BY seat_id) AS grp
  FROM available_seats
)
SELECT MIN(seat_id) AS range_start, MAX(seat_id) AS range_end, COUNT(*) AS seats
FROM numbered
GROUP BY grp
HAVING COUNT(*) >= 2;                  -- only groups of 2+ consecutive seats
```

### §4d — Finding missing values in a sequence

```sql
-- Find missing dates using date series + LEFT JOIN
WITH RECURSIVE date_series AS (
  SELECT MIN(activity_date) AS dt FROM activity
  UNION ALL
  SELECT DATE_ADD(dt, INTERVAL 1 DAY)
  FROM date_series
  WHERE dt < (SELECT MAX(activity_date) FROM activity)
)
SELECT d.dt AS missing_date
FROM date_series d
LEFT JOIN activity a ON d.dt = a.activity_date
WHERE a.activity_date IS NULL;
```

**Common mistakes:**
- Using `RANK()` instead of `ROW_NUMBER()` — ties in dates break the subtraction trick; always use `ROW_NUMBER()`
- Forgetting `PARTITION BY user_id` — groups streaks across all users instead of per user
- Not deduplicating dates first — if a user has multiple events on the same date, duplicate dates break the trick

---
## §5 — Session Grouping

**Signal:** "group events into sessions", "events within N minutes of each other belong to the same session"

This is a generalized Gaps & Islands variant — instead of consecutive dates, you group events where the gap to the previous event is below a threshold.

### §5a — LAG-based session grouping

```sql
-- Step 1: flag new session start (gap > threshold)
WITH flagged AS (
  SELECT
    user_id, event_time,
    CASE
      WHEN TIMESTAMPDIFF(
        MINUTE,
        LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
        event_time
      ) > 30                           -- 30-minute session gap threshold
      OR LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL
      THEN 1 ELSE 0
    END AS is_new_session
  FROM events
),
-- Step 2: assign session ID using cumulative sum of new session flags
sessions AS (
  SELECT
    user_id, event_time,
    SUM(is_new_session) OVER (
      PARTITION BY user_id ORDER BY event_time
    ) AS session_id
  FROM flagged
)
-- Step 3: aggregate per session
SELECT
  user_id,
  session_id,
  MIN(event_time)  AS session_start,
  MAX(event_time)  AS session_end,
  COUNT(*)         AS events_in_session,
  TIMESTAMPDIFF(MINUTE, MIN(event_time), MAX(event_time)) AS session_duration_min
FROM sessions
GROUP BY user_id, session_id
ORDER BY user_id, session_id;
```

**Decision — Gaps & Islands vs Session Grouping:**
| Pattern | Use when |
| :--- | :--- |
| Row number subtraction | Consecutive dates or integers — fixed step size |
| LAG + cumulative SUM | Variable time gaps — group by threshold |

**Common mistakes:**
- Forgetting `OR LAG(...) IS NULL` for the first event — first row has no previous event and should always start a new session
- Using `DATEDIFF` instead of `TIMESTAMPDIFF` — `DATEDIFF` only returns days; use `TIMESTAMPDIFF(MINUTE, ...)` for sub-day gaps
- Applying the session flag without cumulative SUM — a flag alone doesn't assign a unique session ID

---
## §6 — Product Metrics

**Signal:** "daily active users", "retention rate", "cohort", "conversion funnel"

### §6a — DAU / WAU / MAU

```sql
SELECT
  COUNT(DISTINCT CASE
    WHEN event_date >= CURRENT_DATE - INTERVAL 1 DAY  THEN user_id END) AS DAU,
  COUNT(DISTINCT CASE
    WHEN event_date >= CURRENT_DATE - INTERVAL 7 DAY  THEN user_id END) AS WAU,
  COUNT(DISTINCT CASE
    WHEN event_date >= CURRENT_DATE - INTERVAL 30 DAY THEN user_id END) AS MAU
FROM events;
```

### §6b — Cohort retention

```sql
-- Assign each user to their cohort (first activity date)
WITH cohort AS (
  SELECT user_id, MIN(DATE(event_ts)) AS cohort_date
  FROM events
  GROUP BY user_id
),
-- Calculate week number relative to cohort date
activity AS (
  SELECT
    e.user_id,
    c.cohort_date,
    FLOOR(DATEDIFF(DATE(e.event_ts), c.cohort_date) / 7) AS week_num
  FROM events e
  JOIN cohort c ON e.user_id = c.user_id
)
SELECT
  cohort_date,
  week_num,
  COUNT(DISTINCT user_id) AS retained_users
FROM activity
GROUP BY cohort_date, week_num
ORDER BY cohort_date, week_num;
```

### §6c — Funnel analysis

```sql
-- Count distinct users at each step
SELECT
  COUNT(DISTINCT CASE WHEN step = 'view'     THEN user_id END) AS step1_view,
  COUNT(DISTINCT CASE WHEN step = 'cart'     THEN user_id END) AS step2_cart,
  COUNT(DISTINCT CASE WHEN step = 'checkout' THEN user_id END) AS step3_checkout
FROM events;

-- With conversion rates
WITH steps AS (
  SELECT
    COUNT(DISTINCT CASE WHEN step = 'view'     THEN user_id END) AS s1,
    COUNT(DISTINCT CASE WHEN step = 'cart'     THEN user_id END) AS s2,
    COUNT(DISTINCT CASE WHEN step = 'checkout' THEN user_id END) AS s3
  FROM events
)
SELECT
  s1 AS view_users,
  s2 AS cart_users,
  s3 AS checkout_users,
  ROUND(s2 * 100.0 / NULLIF(s1, 0), 2) AS view_to_cart_pct,
  ROUND(s3 * 100.0 / NULLIF(s2, 0), 2) AS cart_to_checkout_pct
FROM steps;
```

**Common mistakes:**
- Funnel: `WHERE step = 'view'` instead of `CASE WHEN` — collapses to only view-step rows
- Cohort: not deduplicating events per day — use `COUNT(DISTINCT user_id)` not `COUNT(*)`
- Division by zero in conversion rates — always wrap denominator in `NULLIF(..., 0)`

---
## §7 — Growth Metrics

**Signal:** "week-over-week change", "month-over-month growth", "rolling 7-day active users"

### §7a — Week-over-week / Month-over-month growth

```sql
-- Weekly active users with WoW growth rate
WITH weekly AS (
  SELECT
    YEARWEEK(event_date)              AS yr_week,
    COUNT(DISTINCT user_id)           AS wau
  FROM events
  GROUP BY YEARWEEK(event_date)
)
SELECT
  yr_week,
  wau,
  LAG(wau) OVER (ORDER BY yr_week)   AS prev_week_wau,
  ROUND(
    (wau - LAG(wau) OVER (ORDER BY yr_week)) * 100.0
    / NULLIF(LAG(wau) OVER (ORDER BY yr_week), 0),
    2
  )                                   AS wow_growth_pct
FROM weekly
ORDER BY yr_week;

-- Month-over-month: replace YEARWEEK with DATE_FORMAT(date, '%Y-%m')
```

### §7b — Rolling N-day active users (sliding window)

```sql
-- Rolling 7-day distinct active users for each date
-- Different from MAU: window slides daily, not fixed from a start date
WITH daily AS (
  SELECT DISTINCT user_id, DATE(event_ts) AS event_date
  FROM events
)
SELECT
  d1.event_date,
  COUNT(DISTINCT d2.user_id) AS rolling_7d_users
FROM
  (SELECT DISTINCT event_date FROM daily) d1
  JOIN daily d2
    ON d2.event_date BETWEEN DATE_SUB(d1.event_date, INTERVAL 6 DAY)
                         AND d1.event_date
GROUP BY d1.event_date
ORDER BY d1.event_date;
```

### §7c — Day-1 / Day-7 retention

```sql
-- For each install date cohort, what fraction returned on day 1 and day 7?
WITH installs AS (
  SELECT player_id, MIN(event_date) AS install_date
  FROM Activity
  GROUP BY player_id
)
SELECT
  ROUND(
    COUNT(DISTINCT CASE
      WHEN a.event_date = DATE_ADD(i.install_date, INTERVAL 1 DAY)
      THEN a.player_id END
    ) * 1.0 / COUNT(DISTINCT i.player_id),
    2
  ) AS day1_retention
FROM installs i
LEFT JOIN Activity a ON i.player_id = a.player_id;
```

**Decision — fixed window vs rolling window:**
| Metric | Window type | Use when |
| :--- | :--- | :--- |
| MAU | Fixed 30-day from today | Snapshot of recent activity |
| Rolling 7d users | Sliding 7-day per date | Trend over time, each date gets its own window |

**Common mistakes:**
- Rolling window via self-join without deduplication — use `DISTINCT user_id` before joining
- WoW growth: using `WEEK()` instead of `YEARWEEK()` — `WEEK()` resets at year boundary causing wrong week 1 pairing
- Day-N retention: off-by-one in `INTERVAL` — day 1 retention = returned exactly 1 day after install

---
## §8 — Recursive Patterns

**Signal:** "hierarchy", "org chart", "all managers of", "generate date series", "find all ancestors"

### §8a — Org chart top-down traversal

```sql
WITH RECURSIVE hierarchy AS (
  -- Anchor: root nodes
  SELECT id, name, manager_id, 0 AS depth
  FROM employees
  WHERE manager_id IS NULL

  UNION ALL

  -- Recursive: one level deeper each iteration
  SELECT e.id, e.name, e.manager_id, h.depth + 1
  FROM employees e
  JOIN hierarchy h ON e.manager_id = h.id
)
SELECT * FROM hierarchy ORDER BY depth, id;
```

### §8b — Find all ancestors (bottom-up)

```sql
WITH RECURSIVE ancestors AS (
  -- Anchor: start from a specific node
  SELECT id, name, manager_id
  FROM employees
  WHERE id = 42

  UNION ALL

  -- Recursive: walk up to parent
  SELECT e.id, e.name, e.manager_id
  FROM employees e
  JOIN ancestors a ON e.id = a.manager_id
)
SELECT * FROM ancestors;
```

### §8c — Date series generation

```sql
-- Generate every date in a range
-- Use case: LEFT JOIN to show zero-activity days
WITH RECURSIVE date_series AS (
  SELECT '2024-01-01' AS dt
  UNION ALL
  SELECT DATE_ADD(dt, INTERVAL 1 DAY)
  FROM date_series
  WHERE dt < '2024-01-31'
)
SELECT d.dt, COALESCE(COUNT(e.user_id), 0) AS event_count
FROM date_series d
LEFT JOIN events e ON DATE(e.event_ts) = d.dt
GROUP BY d.dt
ORDER BY d.dt;
```

### §8d — Number series generation

```sql
-- Generate integers 1 to N
WITH RECURSIVE numbers AS (
  SELECT 1 AS n
  UNION ALL
  SELECT n + 1 FROM numbers WHERE n < 100
)
SELECT * FROM numbers;
```

**Common mistakes:**
- Using `UNION` instead of `UNION ALL` — deduplication breaks the recursion
- Missing termination condition — causes infinite recursion
- Column types in anchor vs recursive step must match exactly

---
# Decision Guide

```
What is the problem asking?
│
├── Remove or identify duplicate rows?
│   ├── All columns identical                → DISTINCT
│   ├── Deduplicate + aggregate              → GROUP BY
│   ├── Keep first / last / nth occurrence   → ROW_NUMBER() + filter
│   ├── Find rows with no match elsewhere    → NOT EXISTS
│   └── Physically delete duplicates         → DELETE + self-join
│
├── Find top N or ranked rows?
│   ├── Exactly N rows, ties broken          → ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)
│   ├── Top N values, ties all included      → DENSE_RANK() OVER (...) WHERE dr <= N
│   └── Single Nth value                     → DENSE_RANK or LIMIT + OFFSET
│
├── Pivot rows into columns?
│   ├── Fixed, known columns                 → SUM(CASE WHEN col = val THEN ... END)
│   └── Dynamic columns                      → Dynamic SQL with GROUP_CONCAT
│
├── Find consecutive sequences or gaps?
│   ├── Fixed step (daily dates, integers)   → date - ROW_NUMBER() grouping trick
│   └── Variable time gap threshold          → LAG + is_new_session flag + cumulative SUM
│
├── Product / user metrics?
│   ├── Active users in a fixed window       → COUNT(DISTINCT) with INTERVAL filter
│   ├── Active users in a sliding window     → Self-join with BETWEEN date range
│   ├── Cohort retention                     → cohort CTE + DATEDIFF / 7 for week_num
│   ├── Funnel conversion                    → COUNT(DISTINCT CASE WHEN step = X)
│   └── WoW / MoM growth                    → LAG over weekly/monthly aggregate
│
├── Hierarchical or recursive data?
│   ├── Top-down traversal                   → Recursive CTE, anchor = root
│   ├── Bottom-up (find ancestors)           → Recursive CTE, anchor = leaf node
│   └── Generate date / number series        → Recursive CTE, anchor = start value
│
├── Window function frame?
│   ├── Last k physical rows                 → ROWS BETWEEN k PRECEDING AND CURRENT ROW
│   ├── All rows in partition                → ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
│   ├── Value-based window (ties together)   → RANGE BETWEEN ... AND CURRENT ROW
│   └── Running total                        → ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
│
├── String matching?
│   ├── Simple contains / starts / ends      → LIKE with % and _
│   ├── Exact substring replacement          → REPLACE()
│   └── Format validation / complex pattern  → REGEXP / REGEXP_LIKE
│
└── Deduplication inside aggregation?
    ├── Count unique values                  → COUNT(DISTINCT col)
    └── Sum only matching rows               → SUM(CASE WHEN ... THEN val ELSE 0 END)
```